In [ ]:
from datetime import timedelta

from snowflake.core import Root
from snowflake.core.task.dagv1 import DAG, DAGTask, DAGOperation
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark import Row
from snowflake.snowpark import types as T

In [ ]:
session = get_active_session()
session.use_schema("TWHITE.PUBLIC")
root = Root(session)

In [ ]:
data = [
    Row(query_id=1, sql_text="SELECT 1 AS A_NUMBER, 'A' AS A_LETTER"),
    Row(query_id=2, sql_text="SELECT 2 AS A_NUMBER"),
    Row(query_id=3, sql_text="SELECT 3 AS A_NUMBER"),
    Row(query_id=4, sql_text="SELECT 4 AS A_NUMBER"),
    Row(query_id=5, sql_text="SELECT 5 AS A_NUMBER"),
]

df = session.create_dataframe(data)
df.write.save_as_table("CONTROL_TABLE", mode="overwrite")

In [ ]:
%%sql -r create_results_table
CREATE OR REPLACE TABLE QUERY_RESULTS (
    QUERY_ID NUMBER,
    QUERY_RESULT VARIANT,
    EXECUTED_AT TIMESTAMP_LTZ DEFAULT CURRENT_TIMESTAMP()
)

In [ ]:
control_rows = session.table("CONTROL_TABLE").collect()

with DAG("CONTROL_DAG", schedule=timedelta(days=1), warehouse="COMPUTE_WH") as dag:
    query_tasks = []
    for row in control_rows:
        qid = row["QUERY_ID"]
        sql = row["SQL_TEXT"]
        task = DAGTask(
            f"QUERY_TASK_{qid}",
            f"""INSERT INTO TWHITE.PUBLIC.QUERY_RESULTS (
                    QUERY_ID,
                    QUERY_RESULT,
                    EXECUTED_AT
                )
                SELECT
                    {qid}                  AS QUERY_ID,
                    OBJECT_CONSTRUCT(*)    AS QUERY_RESULT,
                    CURRENT_TIMESTAMP()    AS EXECUTED_AT
                FROM (
                    {sql}
                )""",
            warehouse="COMPUTE_WH",
        )
        query_tasks.append(task)

    final_task = DAGTask(
        "FINAL_TASK",
        "SELECT 'ALL QUERIES COMPLETE' AS STATUS",
        warehouse="COMPUTE_WH",
    )
    for task in query_tasks:
        task >> final_task

schema = root.databases["TWHITE"].schemas["PUBLIC"]
dag_op = DAGOperation(schema)
dag_op.deploy(dag, mode="orreplace")
print("DAG deployed successfully")

In [ ]:
dag_op.run(dag)
print("DAG execution triggered")

In [ ]:
results = session.sql("SELECT * FROM TWHITE.PUBLIC.QUERY_RESULTS ORDER BY QUERY_ID").collect()
for r in results:
    print(r)

In [ ]:
%%sql -r suspend_dag
ALTER TASK TWHITE.PUBLIC.CONTROL_DAG SUSPEND